In [4]:
from pymilvus import connections
try:
    connections.connect('default', host='localhost', port=19530, timeout=10)
    print('✅ Milvus connection successful')
    connections.disconnect('default')
except Exception as e:
    print(f'❌ Milvus connection failed: {e}')
    exit(1)

✅ Milvus connection successful


In [6]:
# Cell 1: Setup and Imports
import os
import json
import time
import uuid
import logging
import threading
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm import tqdm

# Storage imports
import boto3
from google.cloud import storage as gcs
from botocore.config import Config as BotoConfig

# Milvus imports
from pymilvus import (
    MilvusClient, 
    DataType, 
    Collection, 
    connections, 
    utility
)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ All imports successful")

✅ All imports successful


In [33]:
# Cell 2: Configuration
class Config:
    # Embedding settings
    EMBEDDING_DIR = "./embeddings"  # Directory with NV-Ingest output
    EMBEDDING_DIM = 2048  # Adjust based on your model
    
    # Storage settings
    STORAGE_MODE = "gcp"  # "gcp", "infinia", or "minio"
    
    # GCP settings
    GCP_BUCKET = "infinia-multimodal-milvus"
    GCP_PROJECT = "infinia-solutions-436513"
    
    # Infinia S3 settings (if using)
    # INFINIA_ENDPOINT = "https://your-infinia-endpoint"
    # INFINIA_ACCESS_KEY = os.getenv("INFINIA_ACCESS_KEY", "")
    # INFINIA_SECRET_KEY = os.getenv("INFINIA_SECRET_KEY", "")
    # INFINIA_BUCKET = "your-infinia-bucket"
    # INFINIA_CA_CERT = "/path/to/ca-cert.crt"  # Optional
    
    # MinIO settings (for local testing)
    # MINIO_ENDPOINT = "localhost:9000"
    # MINIO_ACCESS_KEY = "minioadmin"
    # MINIO_SECRET_KEY = "minioadmin"
    # MINIO_BUCKET = "milvus-test"
    
    # Milvus settings
    MILVUS_HOST = "localhost"
    MILVUS_PORT = 19530
    COLLECTION_NAME = "nv_ingest_embeddings"
    RECREATE_COLLECTION = True  # Set to False to reuse existing collection
    
    # Processing settings
    MAX_SEGMENT_ROWS = 240_000  # ~1GB segments (240k * 2048 * 4 bytes)
    BATCH_SIZE = 5000  # For progress updates
    NUM_WORKERS = 8  # Parallel upload workers
    
    # Index settings
    INDEX_TYPE = "HNSW"  # GPU_CAGRA or "HNSW" for CPU 
    
    # Test data settings
    TEST_NUM_EMBEDDINGS = 5000  # Number of test embeddings to generate
    USE_TEST_DATA = True  # Set to False to use real embeddings
    
print(f"Configuration loaded for {Config.STORAGE_MODE} mode")
print(f"Test mode: {'ON' if Config.USE_TEST_DATA else 'OFF'}")
print(f"Collection recreation: {'ON' if Config.RECREATE_COLLECTION else 'OFF'}")

Configuration loaded for gcp mode
Test mode: ON
Collection recreation: ON


In [34]:
# Cell 3: Storage Client Factory
class StorageClient:
    """Factory for creating storage clients"""
    
    @staticmethod
    def create_client(mode: str):
        if mode == "gcp":
            return GCPStorageClient()
        elif mode == "infinia":
            return InfiniaStorageClient()
        elif mode == "minio":
            return MinIOStorageClient()
        else:
            raise ValueError(f"Unknown storage mode: {mode}")


class MinIOStorageClient:
    """MinIO client for local testing"""
    
    def __init__(self):
        from minio import Minio
        
        self.client = Minio(
            Config.MINIO_ENDPOINT,
            access_key=Config.MINIO_ACCESS_KEY,
            secret_key=Config.MINIO_SECRET_KEY,
            secure=False
        )
        self.bucket = Config.MINIO_BUCKET
        
        # Create bucket if not exists
        if not self.client.bucket_exists(self.bucket):
            self.client.make_bucket(self.bucket)
            
        logger.info(f"Initialized MinIO client for bucket: {self.bucket}")
    
    def upload_file(self, local_path: str, remote_path: str):
        """Upload file to MinIO"""
        self.client.fput_object(self.bucket, remote_path, local_path)
        return f"s3://{self.bucket}/{remote_path}"
    
    def list_files(self, prefix: str) -> List[str]:
        """List files with given prefix"""
        objects = self.client.list_objects(self.bucket, prefix=prefix)
        return [obj.object_name for obj in objects]


class GCPStorageClient:
    """GCP Storage client wrapper with better authentication handling"""
    
    def __init__(self):
        import google.auth
        from google.oauth2 import service_account
        
        try:
            # Check for service account key first
            if os.environ.get('GOOGLE_APPLICATION_CREDENTIALS'):
                logger.info(f"Using service account from: {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")
                self.client = gcs.Client.from_service_account_json(
                    os.environ['GOOGLE_APPLICATION_CREDENTIALS'],
                    project=Config.GCP_PROJECT
                )
            else:
                # Try default credentials with explicit scopes
                credentials, project = google.auth.default(
                    scopes=['https://www.googleapis.com/auth/cloud-platform',
                            'https://www.googleapis.com/auth/devstorage.full_control']
                )
                self.client = gcs.Client(
                    project=Config.GCP_PROJECT,
                    credentials=credentials
                )
                logger.info(f"Using default credentials for project: {project}")
            
            self.bucket = self.client.bucket(Config.GCP_BUCKET)
            
            # Test access
            self._test_bucket_access()
            
            logger.info(f"Initialized GCP Storage client for bucket: {Config.GCP_BUCKET}")
            
        except Exception as e:
            logger.error(f"Failed to initialize GCP Storage client: {e}")
            logger.error("Try one of these solutions:")
            logger.error("1. Set GOOGLE_APPLICATION_CREDENTIALS to a service account key")
            logger.error("2. Run: gcloud auth application-default login --scopes=https://www.googleapis.com/auth/cloud-platform")
            logger.error("3. Create a service account with storage.admin role")
            raise
    
    def _test_bucket_access(self):
        """Test if we have write access to the bucket"""
        try:
            # Try to create a test blob
            test_blob = self.bucket.blob('_test_access_check')
            test_blob.upload_from_string('test')
            test_blob.delete()
            logger.info("✓ Bucket write access verified")
        except Exception as e:
            logger.error(f"❌ Bucket write access test failed: {e}")
            if "403" in str(e) and "scope" in str(e).lower():
                logger.error("This is a scope issue. Please re-authenticate with:")
                logger.error("gcloud auth application-default login --scopes=https://www.googleapis.com/auth/cloud-platform")
            raise
    
    def upload_file(self, local_path: str, remote_path: str):
        """Upload file to GCP Storage"""
        blob = self.bucket.blob(remote_path)
        blob.upload_from_filename(local_path)
        return f"gs://{Config.GCP_BUCKET}/{remote_path}"
    
    def list_files(self, prefix: str) -> List[str]:
        """List files with given prefix"""
        return [blob.name for blob in self.bucket.list_blobs(prefix=prefix)]


class InfiniaStorageClient:
    """Infinia S3-compatible storage client wrapper"""
    
    def __init__(self):
        # Configure boto3 for Infinia
        session = boto3.Session()
        
        # SSL configuration
        verify = True
        if Config.INFINIA_CA_CERT and os.path.exists(Config.INFINIA_CA_CERT):
            verify = Config.INFINIA_CA_CERT
            
        self.client = session.client(
            's3',
            endpoint_url=Config.INFINIA_ENDPOINT,
            aws_access_key_id=Config.INFINIA_ACCESS_KEY,
            aws_secret_access_key=Config.INFINIA_SECRET_KEY,
            config=BotoConfig(signature_version='s3v4'),
            verify=verify
        )
        self.bucket = Config.INFINIA_BUCKET
        logger.info(f"Initialized Infinia Storage client for bucket: {self.bucket}")
    
    def upload_file(self, local_path: str, remote_path: str):
        """Upload file to Infinia Storage"""
        self.client.upload_file(local_path, self.bucket, remote_path)
        return f"s3://{self.bucket}/{remote_path}"
    
    def list_files(self, prefix: str) -> List[str]:
        """List files with given prefix"""
        response = self.client.list_objects_v2(
            Bucket=self.bucket,
            Prefix=prefix
        )
        return [obj['Key'] for obj in response.get('Contents', [])]

# Initialize storage client
try:
    storage_client = StorageClient.create_client(Config.STORAGE_MODE)
    print(f"✅ Storage client initialized for {Config.STORAGE_MODE}")
except Exception as e:
    print(f"❌ Failed to initialize storage client: {e}")
    print("Please check your authentication and configuration")

2025-07-04 13:18:51,209 - __main__ - INFO - Using default credentials for project: infinia-solutions-436513
2025-07-04 13:18:51,736 - __main__ - INFO - ✓ Bucket write access verified
2025-07-04 13:18:51,737 - __main__ - INFO - Initialized GCP Storage client for bucket: infinia-multimodal-milvus


✅ Storage client initialized for gcp


In [35]:
# Cell 4: Test Data Generator
class TestDataGenerator:
    """Generate synthetic embeddings for testing"""
    
    def __init__(self, output_dir: str = "./test_embeddings"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
    def generate_test_embeddings(self, 
                               num_embeddings: int = 10000,
                               embedding_dim: int = 2048,
                               batch_size: int = 1000) -> str:
        """Generate random embeddings and metadata for testing"""
        
        logger.info(f"Generating {num_embeddings} test embeddings of dimension {embedding_dim}")
        
        metadata = []
        embeddings_dir = self.output_dir / "embeddings"
        embeddings_dir.mkdir(exist_ok=True)
        
        # Generate embeddings in batches
        for i in tqdm(range(0, num_embeddings, batch_size), desc="Generating test data"):
            batch_end = min(i + batch_size, num_embeddings)
            
            for j in range(i, batch_end):
                # Generate random embedding
                embedding = np.random.randn(embedding_dim).astype(np.float32)
                
                # Normalize to unit length (common for embeddings)
                embedding = embedding / np.linalg.norm(embedding)
                
                # Save embedding
                filename = f"embedding_{j:06d}.npy"
                filepath = embeddings_dir / filename
                np.save(filepath, embedding)
                
                # Create metadata
                meta = {
                    "filename": filename,
                    "filepath": str(filepath),
                    "embedding_dim": embedding_dim,
                    "content": f"This is test document {j}. " + "Lorem ipsum " * 10,
                    "source_file": f"test_doc_{j//100}.pdf",
                    "source_name": f"test_doc_{j//100}.pdf",
                    "page_number": (j % 10) + 1,
                    "chunk_index": j,
                    "collection": "test_collection"
                }
                metadata.append(meta)
        
        # Save metadata
        metadata_file = self.output_dir / "embeddings_metadata.json"
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
            
        logger.info(f"Generated {num_embeddings} test embeddings in {self.output_dir}")
        logger.info(f"Total size: {num_embeddings * embedding_dim * 4 / (1024**3):.2f} GB")
        
        return str(self.output_dir)

# Generate test data if enabled
if Config.USE_TEST_DATA:
    print("🎲 Generating test embeddings...")
    generator = TestDataGenerator()
    test_embedding_dir = generator.generate_test_embeddings(
        num_embeddings=Config.TEST_NUM_EMBEDDINGS,
        embedding_dim=Config.EMBEDDING_DIM
    )
    print(f"✅ Test data generated in: {test_embedding_dir}")
    
    # Update config to use test data
    Config.EMBEDDING_DIR = test_embedding_dir
else:
    print(f"📁 Using real embeddings from: {Config.EMBEDDING_DIR}")

2025-07-04 13:18:53,407 - __main__ - INFO - Generating 5000 test embeddings of dimension 2048


🎲 Generating test embeddings...


Generating test data: 100%|██████████| 5/5 [00:01<00:00,  3.30it/s]
2025-07-04 13:18:54,980 - __main__ - INFO - Generated 5000 test embeddings in test_embeddings
2025-07-04 13:18:54,981 - __main__ - INFO - Total size: 0.04 GB


✅ Test data generated in: test_embeddings


In [36]:
# Cell 5: Embedding Loader and Scanner
class EmbeddingLoader:
    """Load embeddings from directory (works with both real and test data)"""
    
    def __init__(self, embedding_dir: str):
        self.embedding_dir = Path(embedding_dir)
        self.metadata_file = self.embedding_dir / "embeddings_metadata.json"
        
    def scan_embeddings(self) -> Tuple[List[Dict], int]:
        """Scan embedding directory and return metadata and total count"""
        if not self.metadata_file.exists():
            raise FileNotFoundError(f"Metadata file not found: {self.metadata_file}")
            
        with open(self.metadata_file, 'r') as f:
            metadata = json.load(f)
            
        # Validate embeddings exist
        valid_metadata = []
        for meta in metadata:
            if 'filepath' in meta and os.path.exists(meta['filepath']):
                valid_metadata.append(meta)
            else:
                logger.warning(f"Embedding file not found: {meta.get('filepath', 'unknown')}")
                
        logger.info(f"Found {len(valid_metadata)} valid embeddings out of {len(metadata)} total")
        return valid_metadata, len(valid_metadata)
    
    def group_into_segments(self, metadata: List[Dict], max_rows: int) -> List[List[Dict]]:
        """Group embeddings into segments of approximately max_rows size"""
        segments = []
        current_segment = []
        current_size = 0
        
        for meta in metadata:
            current_segment.append(meta)
            current_size += 1
            
            if current_size >= max_rows:
                segments.append(current_segment)
                current_segment = []
                current_size = 0
        
        # Add remaining embeddings
        if current_segment:
            segments.append(current_segment)
            
        logger.info(f"Grouped {len(metadata)} embeddings into {len(segments)} segments")
        return segments

# Test the loader with generated data
loader = EmbeddingLoader(Config.EMBEDDING_DIR)
try:
    metadata, total_count = loader.scan_embeddings()
    print(f"✅ Found {total_count} embeddings")
    
    # Show sample metadata
    if metadata:
        print(f"\nSample metadata entry:")
        print(json.dumps(metadata[0], indent=2))
except Exception as e:
    print(f"❌ Error loading embeddings: {e}")

2025-07-04 13:19:04,516 - __main__ - INFO - Found 5000 valid embeddings out of 5000 total


✅ Found 5000 embeddings

Sample metadata entry:
{
  "filename": "embedding_000000.npy",
  "filepath": "test_embeddings/embeddings/embedding_000000.npy",
  "embedding_dim": 2048,
  "content": "This is test document 0. Lorem ipsum Lorem ipsum Lorem ipsum Lorem ipsum Lorem ipsum Lorem ipsum Lorem ipsum Lorem ipsum Lorem ipsum Lorem ipsum ",
  "source_file": "test_doc_0.pdf",
  "source_name": "test_doc_0.pdf",
  "page_number": 1,
  "chunk_index": 0,
  "collection": "test_collection"
}


In [37]:
# Cell 6: Segment Aggregator (NVIDIA approach)
class SegmentAggregator:
    """Aggregate small embedding files into larger segments"""
    
    def __init__(self, temp_dir: str = "/tmp/milvus_segments"):
        self.temp_dir = Path(temp_dir)
        self.temp_dir.mkdir(parents=True, exist_ok=True)
        
    def aggregate_segment(self, segment_metadata: List[Dict], segment_id: str) -> Tuple[str, str]:
        """
        Aggregate multiple embeddings into single NPY files.
        Returns paths to aggregated id and embedding files.
        """
        segment_dir = self.temp_dir / segment_id
        segment_dir.mkdir(exist_ok=True)
        
        all_ids = []
        all_embeddings = []
        all_texts = []
        all_sources = []
        
        # Load and aggregate embeddings
        for idx, meta in enumerate(tqdm(segment_metadata, desc=f"Aggregating segment {segment_id}")):
            try:
                # Load embedding
                embedding = np.load(meta['filepath'])
                
                # Generate unique integer ID based on segment and index
                # This ensures unique IDs across all segments
                segment_num = int(segment_id.split('_')[-1])
                unique_id = segment_num * 1_000_000 + idx  # Allows up to 1M embeddings per segment
                
                all_ids.append(unique_id)
                all_embeddings.append(embedding)
                all_texts.append(meta.get('content', '')[:65535])  # Truncate for Milvus
                all_sources.append(meta.get('source_name', 'unknown'))
                
            except Exception as e:
                logger.error(f"Error loading embedding: {e}")
                continue
        
        # Convert to numpy arrays with correct dtypes
        ids_array = np.array(all_ids, dtype=np.int64)  # Use int64 for IDs
        embeddings_array = np.array(all_embeddings, dtype=np.float32)
        
        # Save aggregated files
        id_path = segment_dir / "id.npy"
        embedding_path = segment_dir / "embedding.npy"
        metadata_path = segment_dir / "metadata.json"
        
        np.save(id_path, ids_array)
        np.save(embedding_path, embeddings_array)
        
        # Save additional metadata with ID mapping
        segment_info = {
            'num_embeddings': len(all_ids),
            'id_mapping': {str(all_ids[i]): {
                'original_file': segment_metadata[i].get('filename', ''),
                'source': all_sources[i],
                'text_preview': all_texts[i][:100] + '...' if len(all_texts[i]) > 100 else all_texts[i]
            } for i in range(len(all_ids))},
            'segment_id': segment_id,
            'timestamp': time.time()
        }
        with open(metadata_path, 'w') as f:
            json.dump(segment_info, f)
        
        logger.info(f"Aggregated {len(all_ids)} embeddings into segment {segment_id}")
        logger.info(f"Segment size: {embeddings_array.nbytes / (1024**3):.2f} GB")
        logger.info(f"ID dtype: {ids_array.dtype}, Embedding dtype: {embeddings_array.dtype}")
        
        return str(id_path), str(embedding_path)

# Test aggregator
aggregator = SegmentAggregator()
print("✅ Segment aggregator initialized")

✅ Segment aggregator initialized


In [38]:
# Cell 7: Parallel Upload Manager
class ParallelUploadManager:
    """Manage parallel uploads to storage"""
    
    def __init__(self, storage_client, num_workers: int = 8):
        self.storage_client = storage_client
        self.num_workers = num_workers
        self.upload_results = []
        
    def upload_segment(self, segment_id: str, id_path: str, embedding_path: str) -> Dict:
        """Upload a single segment to storage"""
        try:
            start_time = time.time()
            
            # Define remote paths
            remote_id_path = f"milvus_segments/{segment_id}/id.npy"
            remote_embedding_path = f"milvus_segments/{segment_id}/embedding.npy"
            
            # Upload files
            id_url = self.storage_client.upload_file(id_path, remote_id_path)
            embedding_url = self.storage_client.upload_file(embedding_path, remote_embedding_path)
            
            upload_time = time.time() - start_time
            
            result = {
                'segment_id': segment_id,
                'id_url': id_url,
                'embedding_url': embedding_url,
                'remote_paths': [remote_id_path, remote_embedding_path],
                'upload_time': upload_time,
                'status': 'success'
            }
            
            logger.info(f"Uploaded segment {segment_id} in {upload_time:.2f}s")
            return result
            
        except Exception as e:
            logger.error(f"Failed to upload segment {segment_id}: {e}")
            return {
                'segment_id': segment_id,
                'status': 'failed',
                'error': str(e)
            }
    
    def upload_all_segments(self, segments_to_upload: List[Tuple[str, str, str]]) -> List[Dict]:
        """Upload all segments in parallel"""
        logger.info(f"Starting parallel upload of {len(segments_to_upload)} segments")
        
        with ThreadPoolExecutor(max_workers=self.num_workers) as executor:
            # Submit all upload tasks
            futures = []
            for segment_id, id_path, embedding_path in segments_to_upload:
                future = executor.submit(
                    self.upload_segment, 
                    segment_id, 
                    id_path, 
                    embedding_path
                )
                futures.append(future)
            
            # Collect results with progress bar
            results = []
            with tqdm(total=len(futures), desc="Uploading segments") as pbar:
                for future in as_completed(futures):
                    result = future.result()
                    results.append(result)
                    pbar.update(1)
                    
                    if result['status'] == 'success':
                        pbar.set_postfix({'last_time': f"{result['upload_time']:.1f}s"})
        
        successful = sum(1 for r in results if r['status'] == 'success')
        logger.info(f"Upload complete: {successful}/{len(results)} segments uploaded successfully")
        
        return results

# Initialize upload manager
upload_manager = ParallelUploadManager(storage_client, num_workers=Config.NUM_WORKERS)
print("✅ Upload manager initialized")

✅ Upload manager initialized


In [39]:
# Cell 8: Milvus Bulk Inserter
class MilvusBulkInserter:
    """Handle Milvus collection creation and bulk insert"""
    
    def __init__(self, host: str, port: int, collection_name: str):
        self.host = host
        self.port = port
        self.collection_name = collection_name
        self.client = None
        self.collection = None
        
    def connect(self):
        """Connect to Milvus"""
        connections.connect("default", host=self.host, port=self.port)
        self.client = MilvusClient(uri=f"http://{self.host}:{self.port}")
        logger.info(f"Connected to Milvus at {self.host}:{self.port}")
        
    def create_collection(self, embedding_dim: int, recreate: bool = False):
        """Create collection with schema"""
        if utility.has_collection(self.collection_name):
            if recreate:
                utility.drop_collection(self.collection_name)
                logger.info(f"Dropped existing collection: {self.collection_name}")
            else:
                # Check if existing collection has the correct schema
                self.collection = Collection(self.collection_name)
                existing_schema = self.collection.schema
                
                # Check if ID field is INT64
                id_field = next((f for f in existing_schema.fields if f.name == "id"), None)
                if id_field and id_field.dtype != DataType.INT64:
                    logger.warning("Existing collection has incorrect ID field type, recreating...")
                    utility.drop_collection(self.collection_name)
                else:
                    logger.info(f"Using existing collection: {self.collection_name}")
                    return
        
        # Create schema with INT64 ID field
        schema = self.client.create_schema(auto_id=False, enable_dynamic_field=True)
        
        # Add fields
        schema.add_field(
            field_name="id",
            datatype=DataType.INT64,  # Using INT64 for IDs
            is_primary=True
        )
        schema.add_field(
            field_name="embedding",
            datatype=DataType.FLOAT_VECTOR,
            dim=embedding_dim
        )
        
        # Create collection
        self.collection = Collection(self.collection_name, schema)
        logger.info(f"Created collection: {self.collection_name} with INT64 ID field")
        
    def create_index(self, index_type: str = "GPU_CAGRA", sync: bool = True):
        """Create index on collection"""
        index_params = self.client.prepare_index_params()
        
        if index_type == "GPU_CAGRA":
            index_params.add_index(
                field_name="embedding",
                metric_type="L2",
                index_type="GPU_CAGRA",
                index_name="embedding_index",
                params={
                    "intermediate_graph_degree": 128,
                    "graph_degree": 64,
                    "build_algo": "NN_DESCENT"
                }
            )
        else:  # HNSW
            index_params.add_index(
                field_name="embedding",
                metric_type="L2",
                index_type="HNSW",
                index_name="embedding_index",
                params={
                    "M": 32,
                    "efConstruction": 512
                }
            )
        
        self.client.create_index(
            collection_name=self.collection_name,
            index_params=index_params,
            sync=sync  # If True, waits for index creation to complete
        )
        logger.info(f"Started creating {index_type} index (sync={sync})")
        
    def bulk_insert_segments(self, upload_results: List[Dict]) -> List[str]:
        """Perform bulk insert for uploaded segments"""
        job_ids = []
        
        logger.info(f"Starting bulk insert for {len(upload_results)} segments")
        
        for result in upload_results:
            if result['status'] != 'success':
                continue
                
            try:
                # Milvus expects the remote paths
                job_id = utility.do_bulk_insert(
                    collection_name=self.collection_name,
                    files=result['remote_paths']
                )
                job_ids.append(job_id)
                logger.info(f"Started bulk insert job {job_id} for segment {result['segment_id']}")
                
            except Exception as e:
                logger.error(f"Failed to start bulk insert for segment {result['segment_id']}: {e}")
        
        return job_ids
    
    def monitor_bulk_insert(self, job_ids: List[str]):
        """Monitor bulk insert progress"""
        logger.info(f"Monitoring {len(job_ids)} bulk insert jobs")
        
        completed_jobs = set()
        failed_jobs = set()
        
        with tqdm(total=len(job_ids), desc="Bulk insert progress") as pbar:
            while len(completed_jobs) + len(failed_jobs) < len(job_ids):
                for job_id in job_ids:
                    if job_id in completed_jobs or job_id in failed_jobs:
                        continue
                        
                    try:
                        state = utility.get_bulk_insert_state(task_id=job_id)
                        
                        if state.state_name == "Completed":
                            completed_jobs.add(job_id)
                            pbar.update(1)
                            logger.info(f"Job {job_id} completed successfully")
                        elif state.state_name == "Failed":
                            failed_jobs.add(job_id)
                            pbar.update(1)
                            logger.error(f"Job {job_id} failed: {getattr(state, 'failed_reason', 'Unknown')}")
                            
                    except Exception as e:
                        logger.debug(f"Error checking job {job_id}: {e}")
                
                time.sleep(2)  # Poll interval
        
        logger.info(f"Bulk insert complete: {len(completed_jobs)} succeeded, {len(failed_jobs)} failed")
        return completed_jobs, failed_jobs

# Initialize Milvus inserter
milvus_inserter = MilvusBulkInserter(
    Config.MILVUS_HOST, 
    Config.MILVUS_PORT, 
    Config.COLLECTION_NAME
)
print("✅ Milvus inserter initialized")

✅ Milvus inserter initialized


In [40]:
# Cell 9: Main Pipeline Execution
def run_bulk_indexing_pipeline():
    """Execute the complete bulk indexing pipeline"""
    
    pipeline_start = time.time()
    results = {
        'start_time': pipeline_start,
        'embedding_dir': Config.EMBEDDING_DIR,
        'storage_mode': Config.STORAGE_MODE,
        'collection_name': Config.COLLECTION_NAME
    }
    
    try:
        # Step 1: Load embedding metadata
        print("\n📁 Step 1: Loading embeddings...")
        loader = EmbeddingLoader(Config.EMBEDDING_DIR)
        metadata, total_count = loader.scan_embeddings()
        segments = loader.group_into_segments(metadata, Config.MAX_SEGMENT_ROWS)
        
        results['total_embeddings'] = total_count
        results['num_segments'] = len(segments)
        print(f"✅ Loaded {total_count} embeddings in {len(segments)} segments")
        
        # Step 2: Aggregate segments
        print("\n🔄 Step 2: Aggregating segments...")
        aggregator = SegmentAggregator()
        segments_to_upload = []
        
        aggregate_start = time.time()
        for i, segment_metadata in enumerate(segments):
            segment_id = f"segment_{i:04d}"
            id_path, embedding_path = aggregator.aggregate_segment(segment_metadata, segment_id)
            segments_to_upload.append((segment_id, id_path, embedding_path))
        
        results['aggregation_time'] = time.time() - aggregate_start
        print(f"✅ Aggregated {len(segments)} segments in {results['aggregation_time']:.2f}s")
        
        # Step 3: Upload segments to storage
        print("\n☁️  Step 3: Uploading segments to storage...")
        upload_start = time.time()
        upload_manager = ParallelUploadManager(storage_client, Config.NUM_WORKERS)
        upload_results = upload_manager.upload_all_segments(segments_to_upload)
        
        results['upload_time'] = time.time() - upload_start
        results['successful_uploads'] = sum(1 for r in upload_results if r['status'] == 'success')
        print(f"✅ Uploaded {results['successful_uploads']} segments in {results['upload_time']:.2f}s")
        
        # Step 4: Connect to Milvus and create collection
        print("\n🗄️  Step 4: Setting up Milvus collection...")
        milvus_setup_start = time.time()
        milvus_inserter.connect()
        
        # Create collection with configured recreate flag
        if Config.RECREATE_COLLECTION:
            print("ℹ️  Recreating collection (RECREATE_COLLECTION=True)...")
        else:
            print("ℹ️  Using existing collection if available (RECREATE_COLLECTION=False)...")
            
        milvus_inserter.create_collection(Config.EMBEDDING_DIM, recreate=Config.RECREATE_COLLECTION)
        
        # Verify collection schema
        collection = Collection(Config.COLLECTION_NAME)
        schema_info = collection.schema
        id_field = next((f for f in schema_info.fields if f.name == "id"), None)
        if id_field:
            print(f"✅ ID field type: {id_field.dtype} (should be DataType.INT64={DataType.INT64})")
            if id_field.dtype != DataType.INT64:
                print("⚠️  WARNING: ID field is not INT64! Set RECREATE_COLLECTION=True to fix.")
        
        results['collection_setup_time'] = time.time() - milvus_setup_start
        
        # Step 5: Perform bulk insert FIRST (following NVIDIA's approach)
        print("\n📥 Step 5: Performing bulk insert...")
        ingestion_start = time.time()
        
        job_ids = milvus_inserter.bulk_insert_segments(upload_results)
        completed_jobs, failed_jobs = milvus_inserter.monitor_bulk_insert(job_ids)
        
        results['ingestion_time'] = time.time() - ingestion_start
        results['completed_jobs'] = len(completed_jobs)
        results['failed_jobs'] = len(failed_jobs)
        
        # Step 6: Create index AFTER ingestion (NVIDIA's approach)
        print("\n🔧 Step 6: Creating index on ingested data...")
        index_start = time.time()
        
        # Get total entity count before indexing
        total_entities = milvus_inserter.collection.num_entities
        print(f"ℹ️  Total entities in collection: {total_entities:,}")
        
        milvus_inserter.create_index(Config.INDEX_TYPE, sync=False)
        
        # Wait for index to be built with detailed monitoring
        print("ℹ️  Monitoring index build progress...")
        print("-" * 80)
        
        last_log_time = time.time()
        log_interval = 10  # Log every 10 seconds
        check_count = 0
        
        while True:
            try:
                # Check if index exists
                indexes = milvus_inserter.client.list_indexes(collection_name=Config.COLLECTION_NAME)
                if "embedding_index" not in indexes:
                    if check_count == 0:
                        print("   Waiting for index creation to start...")
                    time.sleep(2)
                    check_count += 1
                    continue
                
                # Get index info
                index_info = milvus_inserter.client.describe_index(
                    collection_name=Config.COLLECTION_NAME,
                    index_name="embedding_index"
                )
                
                if isinstance(index_info, dict):
                    # Extract index state and progress
                    state = index_info.get("state", "Unknown")
                    indexed_rows = index_info.get("indexed_rows", 0)
                    pending_rows = index_info.get("pending_index_rows", 0)
                    total_rows = indexed_rows + pending_rows
                    
                    # Calculate progress
                    if total_rows > 0:
                        progress = (indexed_rows / total_rows) * 100
                    else:
                        progress = 0
                    
                    # Format state string
                    if isinstance(state, int):
                        state_map = {1: "Failed", 2: "Building", 6: "Finished"}
                        state_str = state_map.get(state, f"Unknown({state})")
                    else:
                        state_str = str(state)
                    
                    # Progress bar
                    bar_length = 40
                    filled_length = int(bar_length * progress / 100)
                    bar = '█' * filled_length + '░' * (bar_length - filled_length)
                    
                    # Update progress display
                    elapsed = time.time() - index_start
                    status_line = (f"\r   [{bar}] {progress:6.2f}% | "
                                  f"State: {state_str:10} | "
                                  f"Indexed: {indexed_rows:,}/{total_rows:,} | "
                                  f"Pending: {pending_rows:,} | "
                                  f"Elapsed: {elapsed:.1f}s")
                    print(status_line, end='', flush=True)
                    
                    # Periodic detailed logs
                    current_time = time.time()
                    if current_time - last_log_time >= log_interval:
                        # Print a newline to preserve the progress bar
                        print()
                        logger.info(f"Index build progress: {progress:.1f}% - "
                                   f"Indexed: {indexed_rows:,}, Pending: {pending_rows:,}, "
                                   f"State: {state_str}")
                        
                        # Estimate time remaining if possible
                        if indexed_rows > 0 and progress < 100:
                            rate = indexed_rows / elapsed
                            remaining_rows = total_rows - indexed_rows
                            eta = remaining_rows / rate
                            logger.info(f"Indexing rate: {rate:.0f} rows/sec, "
                                       f"Estimated time remaining: {eta:.0f}s")
                        
                        last_log_time = current_time
                    
                    # Check if finished
                    if (state_str == "Finished" or state == 6) and pending_rows == 0:
                        print()  # New line after progress bar
                        print("-" * 80)
                        print(f"✅ Index built successfully!")
                        print(f"   Total rows indexed: {indexed_rows:,}")
                        print(f"   Index build time: {elapsed:.2f}s")
                        print(f"   Average rate: {indexed_rows/elapsed:.0f} rows/sec")
                        logger.info(f"Index build completed: {indexed_rows:,} rows in {elapsed:.2f}s")
                        break
                    
                    # Check for failure
                    if state_str == "Failed" or state == 1:
                        print()  # New line after progress bar
                        print(f"❌ Index build failed!")
                        logger.error(f"Index build failed at {indexed_rows:,} rows")
                        break
                
                time.sleep(1)  # Check every second for smooth progress updates
                check_count += 1
                
                # Timeout after 30 minutes
                if time.time() - index_start > 1800:
                    print()  # New line after progress bar
                    print("⚠️  Index build timeout (30 minutes)")
                    logger.warning("Index build timeout reached")
                    break
                    
            except Exception as e:
                if check_count % 10 == 0:  # Log errors every 10 checks
                    logger.debug(f"Error checking index status: {e}")
                time.sleep(2)
                check_count += 1
        
        results['index_creation_time'] = time.time() - index_start
        
        # Calculate total Milvus operations time
        results['milvus_total_time'] = time.time() - milvus_setup_start
        
        # Calculate final metrics
        results['total_time'] = time.time() - pipeline_start
        results['embeddings_per_second'] = total_count / results['total_time']
        
        # Print summary
        print("\n" + "="*60)
        print("🎉 BULK INDEXING COMPLETE")
        print("="*60)
        print(f"Total embeddings: {results['total_embeddings']:,}")
        print(f"Number of segments: {results['num_segments']}")
        print(f"Successful uploads: {results['successful_uploads']}")
        print(f"Completed insert jobs: {results['completed_jobs']}")
        print(f"Failed insert jobs: {results['failed_jobs']}")
        print("-"*60)
        print("⏱️  PERFORMANCE METRICS:")
        print(f"Aggregation time: {results['aggregation_time']:.2f}s")
        print(f"Upload time: {results['upload_time']:.2f}s")
        print(f"Collection setup time: {results.get('collection_setup_time', 0):.2f}s")
        print(f"🔥 Index creation time: {results.get('index_creation_time', 0):.2f}s")
        print(f"🔥 Ingestion time: {results.get('ingestion_time', 0):.2f}s")
        print(f"Total Milvus time: {results.get('milvus_total_time', 0):.2f}s")
        print(f"Total pipeline time: {results['total_time']:.2f}s")
        print("-"*60)
        print("📊 THROUGHPUT:")
        print(f"Overall: {results['embeddings_per_second']:.0f} embeddings/second")
        if results.get('ingestion_time', 0) > 0:
            ingestion_throughput = results['total_embeddings'] / results['ingestion_time']
            print(f"Ingestion only: {ingestion_throughput:.0f} embeddings/second")
        print("="*60)
        
        # Save results
        results_file = Path(Config.EMBEDDING_DIR) / "bulk_indexing_results.json"
        with open(results_file, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n📊 Results saved to: {results_file}")
        
        return results
        
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        raise

# Run the pipeline
if __name__ == "__main__":
    print("🚀 Starting bulk indexing pipeline...")
    results = run_bulk_indexing_pipeline()

2025-07-04 13:19:35,779 - __main__ - INFO - Found 5000 valid embeddings out of 5000 total
2025-07-04 13:19:35,781 - __main__ - INFO - Grouped 5000 embeddings into 1 segments


🚀 Starting bulk indexing pipeline...

📁 Step 1: Loading embeddings...
✅ Loaded 5000 embeddings in 1 segments

🔄 Step 2: Aggregating segments...


Aggregating segment segment_0000:  16%|█▌        | 800/5000 [00:00<00:00, 7999.11it/s]

Aggregating segment segment_0000: 100%|██████████| 5000/5000 [00:00<00:00, 8716.01it/s]
2025-07-04 13:19:36,455 - __main__ - INFO - Aggregated 5000 embeddings into segment segment_0000
2025-07-04 13:19:36,456 - __main__ - INFO - Segment size: 0.04 GB
2025-07-04 13:19:36,456 - __main__ - INFO - ID dtype: int64, Embedding dtype: float32
2025-07-04 13:19:36,462 - __main__ - INFO - Starting parallel upload of 1 segments


✅ Aggregated 1 segments in 0.68s

☁️  Step 3: Uploading segments to storage...


...

✅ Uploaded 1 segments in 1.72s

🗄️  Step 4: Setting up Milvus collection...
ℹ️  Recreating collection (RECREATE_COLLECTION=True)...
✅ ID field type: 5 (should be DataType.INT64=5)

📥 Step 5: Performing bulk insert...


Bulk insert progress: 100%|██████████| 1/1 [00:38<00:00, 38.05s/it]
2025-07-04 13:20:16,286 - __main__ - INFO - Bulk insert complete: 1 succeeded, 0 failed
2025-07-04 13:20:16,297 - __main__ - INFO - Started creating HNSW index (sync=False)



🔧 Step 6: Creating index on ingested data...
ℹ️  Total entities in collection: 5,000
ℹ️  Monitoring index build progress...
--------------------------------------------------------------------------------
   [████████████████████████████████████████] 100.00% | State: Finished   | Indexed: 5,000/5,000 | Pending: 0 | Elapsed: 9.1s

2025-07-04 13:20:25,377 - __main__ - INFO - Index build completed: 5,000 rows in 9.09s



--------------------------------------------------------------------------------
✅ Index built successfully!
   Total rows indexed: 5,000
   Index build time: 9.09s
   Average rate: 550 rows/sec

🎉 BULK INDEXING COMPLETE
Total embeddings: 5,000
Number of segments: 1
Successful uploads: 1
Completed insert jobs: 1
Failed insert jobs: 0
------------------------------------------------------------
⏱️  PERFORMANCE METRICS:
Aggregation time: 0.68s
Upload time: 1.72s
Collection setup time: 0.05s
🔥 Index creation time: 9.09s
🔥 Ingestion time: 38.06s
Total Milvus time: 47.19s
Total pipeline time: 49.64s
------------------------------------------------------------
📊 THROUGHPUT:
Overall: 101 embeddings/second
Ingestion only: 131 embeddings/second

📊 Results saved to: test_embeddings/bulk_indexing_results.json
